# Label consistency ordinal v2

Compare original ordinal score labels in `ranks_1.json` against the v2 relabeling export in `ranks_2.json`.

In [1]:
import json
from pathlib import Path

import pandas as pd
from dlordinal.metrics import amae as amae_score
from IPython.display import display
from sklearn.metrics import cohen_kappa_score

DATA_DIR = Path("data/label-consistency-test/ordinal-v2")
RANKS_1_PATH = DATA_DIR / "ranks_1.json"
RANKS_2_PATH = DATA_DIR / "ranks_2.json"

pd.set_option("display.max_colwidth", 120)

In [2]:
def clip_name(uri: str) -> str:
    return Path(uri.split("://", 1)[-1]).name


def extract_rating(task: dict) -> int | None:
    for annotation in task.get("annotations", []):
        for result in annotation.get("result", []):
            if result.get("type") == "rating" and result.get("from_name") == "execution_score":
                return result.get("value", {}).get("rating")
    return None


ranks_1 = json.loads(RANKS_1_PATH.read_text())
ranks_2 = json.loads(RANKS_2_PATH.read_text())

original_by_clip = {
    clip_name(row["video"]): int(row["execution_score"])
    for row in ranks_1
    if row.get("video") and row.get("execution_score") is not None
}

pairs = []
missing_original = []
for row in ranks_2:
    sampled_video_uri = row.get("data", {}).get("video")
    if not sampled_video_uri:
        continue
    name = clip_name(sampled_video_uri)
    sampled_score = extract_rating(row)
    original_score = original_by_clip.get(name)
    if original_score is None:
        missing_original.append(name)
        continue
    pairs.append(
        {
            "clip_name": name,
            "original_score": original_score,
            "sampled_score": sampled_score,
            "absolute_score_delta": abs(original_score - sampled_score),
            "exact_match": original_score == sampled_score,
        }
    )

pairs_df = pd.DataFrame(pairs)
print(f"ranks_1 rows: {len(ranks_1)}")
print(f"ranks_2 rows: {len(ranks_2)}")
print(f"matched rows: {len(pairs_df)}")
print(f"missing original rows: {len(missing_original)}")
display(pairs_df.head())

ranks_1 rows: 94
ranks_2 rows: 100
matched rows: 94
missing original rows: 6


,clip_name,original_score,sampled_score,absolute_score_delta,exact_match
0,25-10-31 19-46-26 5688-00.02.44.697-00.02.49.504-seg09.mp4,2,2,0,True
1,25-10-31 19-46-26 5688-00.04.33.222-00.04.38.667-seg12.mp4,2,2,0,True
2,25-10-31 19-54-19 5689-00.11.10.367-00.11.15.259-seg20.mp4,2,2,0,True
3,25-10-31 19-54-19 5689-00.13.09.900-00.13.15.589-seg24.mp4,2,2,0,True
4,25-11-02 19-14-55 5695-00.00.12.033-00.00.16.478-seg02.mp4,2,2,0,True


In [3]:
metrics_df = pd.DataFrame(
    [
        {"metric": "matched_rows", "value": len(pairs_df)},
        {"metric": "exact_agreement", "value": float(pairs_df["exact_match"].mean())},
        {"metric": "mean_absolute_score_delta", "value": float(pairs_df["absolute_score_delta"].mean())},
        {"metric": "amae", "value": float(amae_score(pairs_df["original_score"], pairs_df["sampled_score"]))},
        {
            "metric": "cohen_kappa_unweighted",
            "value": float(cohen_kappa_score(pairs_df["original_score"], pairs_df["sampled_score"])),
        },
        {
            "metric": "cohen_kappa_weighted_quadratic",
            "value": float(
                cohen_kappa_score(
                    pairs_df["original_score"],
                    pairs_df["sampled_score"],
                    weights="quadratic",
                )
            ),
        },
    ]
)

confusion_df = pd.crosstab(
    pairs_df["original_score"],
    pairs_df["sampled_score"],
    rownames=["original_score"],
    colnames=["sampled_score"],
    dropna=False,
)

display(metrics_df)
display(confusion_df)

,metric,value
0,matched_rows,94.000000
1,exact_agreement,0.914894
2,mean_absolute_score_delta,0.085106
3,amae,0.064373
4,cohen_kappa_unweighted,0.844048
5,cohen_kappa_weighted_quadratic,0.879680


sampled_score,1,2,3
original_score,,,
1,7,0,0
2,0,49,4
3,0,4,30
